# Multiclass Logistic Regression

## 1. Theory
Multiclass Logistic Regression generalizes standard logistic regression (binary classification) to multi-class problems. The standard approach is to use the Softmax function instead of the Sigmoid function.

The Softmax function calculates probabilities for $K$ classes:
$$\ P(y=c | \mathbf{x}, \mathbf{W}) = \frac{e^{\mathbf{x} \mathbf{W}_c}}{\sum_{k=1}^K e^{\mathbf{x} \mathbf{W}_k}} \$$

Where $\mathbf{W}$ is a weight matrix of size $(features, classes)$.

## 2. Implementation
We use the underlying `LogisticRegression` class but extend or wrap it if needed. For true multinomial logistic regression, cross-entropy loss is minimized over $K$ classes.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from src.ml_from_scratch.linear import LogisticRegression
from src.ml_from_scratch.utils import softmax, accuracy
from sklearn.datasets import make_classification

# NOTE: For a real package, you would modify src.ml_from_scratch.linear.LogisticRegression 
# to directly support softmax or one-vs-rest internally.
np.random.seed(42)
X, y = make_classification(n_samples=300, n_features=2, n_informative=2, n_redundant=0, n_clusters_per_class=1, n_classes=3, seed=8)

plt.scatter(X[:, 0], X[:, 1], c=y, cmap='viridis')
plt.title('Synthetic 3-Class Dataset')
plt.show()

## 3. Training One-vs-Rest (OVR)
Since our base `LogisticRegression` is binary, we can train one classifier per class.

In [ ]:
classes = np.unique(y)
models = {}

# Train one model per class (One-Versus-Rest)
for c in classes:
    y_binary = np.where(y == c, 1, 0)
    model = LogisticRegression(learning_rate=0.1, n_iterations=1000)
    model.fit(X, y_binary)
    models[c] = model
    print(f'Trained OVR model for class {c}')

## 4. Example Use & Visualization
Predicting class by taking the argmax of probabilities across all classifiers.

In [ ]:
def predict_ovr(X_test, models):
    # Get probabilities from all models
    probs = np.zeros((X_test.shape[0], len(models)))
    for c, model in models.items():
        # Our LogisticRegression might not have predict_proba natively exposed,
        # but it uses probability internally. For simplicity, we manually compute it:
        z = np.dot(X_test, model.weights) + model.bias
        from src.ml_from_scratch.utils import sigmoid
        probs[:, c] = sigmoid(z)
    
    # Normalize (pseudo-softmax for OVR)
    return np.argmax(probs, axis=1)

# Calculate accuracy
preds = predict_ovr(X, models)
print(f'OVR Accuracy: {accuracy(y, preds):.2%}')

# Visualizing decision boundary
x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.05), np.arange(y_min, y_max, 0.05))

Z = predict_ovr(np.c_[xx.ravel(), yy.ravel()], models)
Z = Z.reshape(xx.shape)

plt.contourf(xx, yy, Z, alpha=0.3, cmap='viridis')
plt.scatter(X[:, 0], X[:, 1], c=y, edgecolor='k', cmap='viridis')
plt.title('Decision Boundaries for OVR Logistic Regression')
plt.show()